# Predictive Risk Model

Datasets from https://www.openml.org/search?type=data&sort=runs&id=41214&status=active:
- freMTPL2freq
- freMTPL2sev

In [3]:
import sqlite3
import statistics
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import TweedieRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shap
import joblib
from xgboost import plot_importance
import requests

## Setting up Database

In [5]:
conn = sqlite3.connect("../data/risk_model.db")
cursor = conn.cursor()

In [6]:
cursor.execute("DROP TABLE IF EXISTS claims;")
cursor.execute("DROP TABLE IF EXISTS policies;")
cursor.execute('''CREATE TABLE policies (
                    IDpol INTEGER PRIMARY KEY,
                    ClaimNb INTEGER NOT NULL,
                    exposure FLOAT NOT NULL,
                    Area TEXT NOT NULL,
                    VehPower INTEGER NOT NULL,
                    VehAge INTEGER NOT NULL,
                    DrivAge INTEGER NOT NULL,
                    BonusMalus INTEGER NOT NULL,
                    VehBrand VARCHAR(50) NOT NULL,
                    VehGas TEXT NOT NULL,
                    Density INTEGER NOT NULL,
                    Region VARCHAR(50)
                );'''
              )
cursor.execute('''CREATE TABLE claims(
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    IDpol INTEGER NOT NULL,
                    ClaimAmount FLOAT NOT NULL,
                    FOREIGN KEY (IDpol) REFERENCES policies(IDpol)
                );'''
              )

#### Reading Datasets Into Pandas and SQL database

In [8]:
df_policies = pd.read_csv("../data/freMTPL2freq.csv")
df_claims = pd.read_csv("../data/freMTPL2sev.csv")
df_policies.to_sql('policies', conn, if_exists='append', index=False)
df_claims.to_sql('claims', conn, if_exists='append', index=False)

26639

In [9]:
cursor.executescript('''DROP VIEW IF EXISTS risk_model_view;
                CREATE VIEW IF NOT EXISTS risk_model_view AS
                SELECT p.*, COALESCE(SUM(c.ClaimAmount), 0) AS TotalClaimAmount
                    FROM policies p
                    LEFT JOIN claims c ON p.IDpol = c.IDpol
                    GROUP BY p.IDpol'''
              )

In [10]:
df_final = pd.read_sql_query("SELECT * FROM risk_model_view", conn)

In [11]:
df_final.head()

,IDpol,ClaimNb,exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,TotalClaimAmount
0,1,1,0.10,D,5,0,55,50,B12,'Regular',1217,R82,0.0
1,3,1,0.77,D,5,0,55,50,B12,'Regular',1217,R82,0.0
2,5,1,0.75,B,6,2,52,50,B12,'Diesel',54,R22,0.0
3,10,1,0.09,B,7,0,46,50,B12,'Diesel',76,R72,0.0
4,11,1,0.84,B,7,0,46,50,B12,'Diesel',76,R72,0.0


## Encoding Categorical Variables

#### Area

In [14]:
df_final["Area"].unique()

<ArrowStringArray>
['D', 'B', 'E', 'C', 'F', 'A']
Length: 6, dtype: str

From the freMTPL2freq dataset documentation:
- Area: The density value of the city community where the car driver lives in: from "A" for rural area
to "F" for urban centre.

In [16]:
# checking that what the documentation says actually makes sense
df_final.groupby("Area")["Density"].mean()

Area
A       27.502852
B       72.582184
C      248.935324
D     1075.898493
E     4380.299598
F    22014.560878
Name: Density, dtype: float64

In [17]:
encode_area_dict = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6}
df_final["Area"] = df_final["Area"].map(encode_area_dict)

# check mapping worked as expected
df_final.head()

,IDpol,ClaimNb,exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,TotalClaimAmount
0,1,1,0.10,4,5,0,55,50,B12,'Regular',1217,R82,0.0
1,3,1,0.77,4,5,0,55,50,B12,'Regular',1217,R82,0.0
2,5,1,0.75,2,6,2,52,50,B12,'Diesel',54,R22,0.0
3,10,1,0.09,2,7,0,46,50,B12,'Diesel',76,R72,0.0
4,11,1,0.84,2,7,0,46,50,B12,'Diesel',76,R72,0.0


#### VehBrand, VehGas, Region

In [19]:
df_final["VehBrand"].unique()

<ArrowStringArray>
['B12', 'B6', 'B3', 'B2', 'B5', 'B10', 'B14', 'B13', 'B4', 'B1', 'B11']
Length: 11, dtype: str

In [20]:
# using the gt dummies method t make columns for each brand, gas type, and region
df_final = pd.get_dummies(data = df_final, columns=["VehBrand", "VehGas", "Region"], drop_first = True)
df_final.head()

,IDpol,ClaimNb,exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,Density,TotalClaimAmount,...,Region_R53,Region_R54,Region_R72,Region_R73,Region_R74,Region_R82,Region_R83,Region_R91,Region_R93,Region_R94
0,1,1,0.10,4,5,0,55,50,1217,0.0,...,False,False,False,False,False,True,False,False,False,False
1,3,1,0.77,4,5,0,55,50,1217,0.0,...,False,False,False,False,False,True,False,False,False,False
2,5,1,0.75,2,6,2,52,50,54,0.0,...,False,False,False,False,False,False,False,False,False,False
3,10,1,0.09,2,7,0,46,50,76,0.0,...,False,False,True,False,False,False,False,False,False,False
4,11,1,0.84,2,7,0,46,50,76,0.0,...,False,False,True,False,False,False,False,False,False,False


In [21]:
df_final.columns

Index(['IDpol', 'ClaimNb', 'exposure', 'Area', 'VehPower', 'VehAge', 'DrivAge',
       'BonusMalus', 'Density', 'TotalClaimAmount', 'VehBrand_B10',
       'VehBrand_B11', 'VehBrand_B12', 'VehBrand_B13', 'VehBrand_B14',
       'VehBrand_B2', 'VehBrand_B3', 'VehBrand_B4', 'VehBrand_B5',
       'VehBrand_B6', 'VehGas_'Regular'', 'Region_R21', 'Region_R22',
       'Region_R23', 'Region_R24', 'Region_R25', 'Region_R26', 'Region_R31',
       'Region_R41', 'Region_R42', 'Region_R43', 'Region_R52', 'Region_R53',
       'Region_R54', 'Region_R72', 'Region_R73', 'Region_R74', 'Region_R82',
       'Region_R83', 'Region_R91', 'Region_R93', 'Region_R94'],
      dtype='str')

## Setting up the Model

#### Splitting data

In [24]:
y = df_final["TotalClaimAmount"]
X = df_final.drop(columns = ["TotalClaimAmount", "IDpol"])

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_glm = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_glm = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

print(X_train.shape[0] / (X_train.shape[0] + X_test.shape[0]))

0.7999994100408104


### Model Fitting

In [27]:
# intializing models
glm_model = TweedieRegressor(power = 1.5)
xgb_model = XGBRegressor(random_state=42)

In [28]:
#fitting models
glm_model.fit(X_train_glm, y_train)
xgb_model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [29]:
glm_predictions = glm_model.predict(X_test_glm)
xgb_predictions = xgb_model.predict(X_test)
print(f"GLM Mean Absolute Error: {mean_absolute_error(y_test, glm_predictions)}")
print(f"XGB Mean Absolute Error: {mean_absolute_error(y_test, xgb_predictions)}")

GLM Mean Absolute Error: 114.3055413127627
XGB Mean Absolute Error: 158.8119295864896


### Fine Tuning the models

In [31]:
glm_param_grid = {
    'power': [1.2, 1.5, 1.8],
    'alpha': [0.1, 1.0, 10.0]
}

glm_grid = GridSearchCV(
    estimator=TweedieRegressor(link='log'), 
    param_grid=glm_param_grid, 
    scoring='neg_mean_absolute_error', 
    cv=3,
    n_jobs=1
)

glm_grid.fit(X_train_glm, y_train)

print(f"Best GLM Parameters: {glm_grid.best_params_}")

Best GLM Parameters: {'alpha': 0.1, 'power': 1.2}


In [32]:
best_glm = glm_grid.best_estimator_

In [33]:
xgb_param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200]
}

xgb_grid = GridSearchCV(
    estimator=XGBRegressor(objective='reg:tweedie'), 
    param_grid=xgb_param_grid, 
    scoring='neg_mean_absolute_error', 
    cv=3,
    n_jobs=1
)

xgb_grid.fit(X_train, y_train)

print(f"Best XGBoost Parameters: {xgb_grid.best_params_}")

Best XGBoost Parameters: {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 200}


In [34]:
best_xgb = xgb_grid.best_estimator_

In [35]:
glm_predictions = best_glm.predict(X_test_glm)
xgb_predictions = best_xgb.predict(X_test)

# GLM Metrics
print("--- GLM Metrics ---")
print(f"MAE: {mean_absolute_error(y_test, glm_predictions):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, glm_predictions)):.2f}")
print(f"R-squared: {r2_score(y_test, glm_predictions):.4f}")

# XGBoost Metrics
print("\n--- XGBoost Metrics ---")
print(f"MAE: {mean_absolute_error(y_test, xgb_predictions):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, xgb_predictions)):.2f}")
print(f"R-squared: {r2_score(y_test, xgb_predictions):.4f}")

--- GLM Metrics ---
MAE: 137.77
RMSE: 2856.74
R-squared: 0.0024

--- XGBoost Metrics ---
MAE: 72.87
RMSE: 2838.18
R-squared: 0.0154


## Analyzing Key Attributes

In [37]:
feature_importance = pd.DataFrame()
feature_importance["Feature"] = X.columns

feature_importance["Importance"] = best_xgb.feature_importances_

feature_importance = feature_importance.sort_values("Importance", ascending=False)
print(feature_importance)

             Feature  Importance
0            ClaimNb    0.974088
5            DrivAge    0.006631
10      VehBrand_B12    0.003121
4             VehAge    0.002547
14       VehBrand_B3    0.001631
26        Region_R41    0.001486
1           exposure    0.001205
18  VehGas_'Regular'    0.001200
6         BonusMalus    0.001017
23        Region_R25    0.000986
39        Region_R94    0.000820
38        Region_R93    0.000630
3           VehPower    0.000595
34        Region_R74    0.000560
25        Region_R31    0.000527
22        Region_R24    0.000507
15       VehBrand_B4    0.000502
7            Density    0.000307
33        Region_R73    0.000235
29        Region_R52    0.000198
30        Region_R53    0.000171
37        Region_R91    0.000155
20        Region_R22    0.000146
16       VehBrand_B5    0.000143
9       VehBrand_B11    0.000117
17       VehBrand_B6    0.000115
31        Region_R54    0.000078
2               Area    0.000077
13       VehBrand_B2    0.000072
24        

In [39]:
zero_cols = feature_importance[feature_importance['Importance'] == 0]['Feature'].tolist()
X_train_clean = X_train.drop(columns=zero_cols, errors='ignore')
X_test_clean = X_test.drop(columns=zero_cols, errors='ignore')


#ensuring columns still match
X_test_clean = X_test_clean.reindex(columns=X_train_clean.columns, fill_value=0)

best_xgb.fit(X_train_clean, y_train)
zero_cols

['Region_R83',
 'Region_R23',
 'Region_R21',
 'Region_R43',
 'Region_R42',
 'VehBrand_B14',
 'VehBrand_B13']

In [43]:
feature_importance = pd.DataFrame()
feature_importance["Feature"] = X_train_clean.columns
feature_importance["Importance"] = best_xgb.feature_importances_
feature_importance = feature_importance.sort_values("Importance", ascending=False)

xgb_predictions = best_xgb.predict(X_test_clean)

print("\n--- XGBoost Metrics ---")
print(f"MAE: {mean_absolute_error(y_test, xgb_predictions):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, xgb_predictions)):.2f}")
print(f"R-squared: {r2_score(y_test, xgb_predictions):.4f}\n")
print(feature_importance)


--- XGBoost Metrics ---
MAE: 72.87
RMSE: 2838.18
R-squared: 0.0154

             Feature  Importance
0            ClaimNb    0.974088
5            DrivAge    0.006631
10      VehBrand_B12    0.003121
4             VehAge    0.002547
12       VehBrand_B3    0.001631
22        Region_R41    0.001486
1           exposure    0.001205
16  VehGas_'Regular'    0.001200
6         BonusMalus    0.001017
19        Region_R25    0.000986
32        Region_R94    0.000820
31        Region_R93    0.000630
3           VehPower    0.000595
28        Region_R74    0.000560
21        Region_R31    0.000527
18        Region_R24    0.000507
13       VehBrand_B4    0.000502
7            Density    0.000307
27        Region_R73    0.000235
23        Region_R52    0.000198
24        Region_R53    0.000171
30        Region_R91    0.000155
17        Region_R22    0.000146
14       VehBrand_B5    0.000143
9       VehBrand_B11    0.000117
15       VehBrand_B6    0.000115
25        Region_R54    0.000078
2      

## Preparing Data for Flask Server

In [46]:
# Finding reasonable ceiling percentile for risk score
np.percentile(xgb_predictions, 99.87)

1494.9276123046875

In [50]:
baseline_applicant = X_train_clean.median().to_dict()
baseline_applicant['exposure'] = 1.0

baseline_prediction = xgb_predictions.mean()

ceiling_prediction = np.percentile(xgb_predictions, 99.87)

joblib.dump(ceiling_prediction, "../production_api/ceiling_prediction.pkl")
joblib.dump(baseline_prediction, "../production_api/baseline_prediction.pkl")
joblib.dump(baseline_applicant, "../production_api/baseline_applicant.pkl")
joblib.dump(best_xgb, "../production_api/xgb_risk_model.pkl")

['../production_api/xgb_risk_model.pkl']

In [58]:
url = "http://127.0.0.1:5000/predict"
applicant_data = {"ClaimNb": 1, "DrivAge": 25}

response = requests.post(url, json = applicant_data)
print(f"Status Code: {response.status_code}")
print(f"Server Output: {response.text}")

Status Code: 200
Server Output: {
  "status": "success",
  "Predicted Claim Amount_USD": 697.22,
  "Risk Severity Score 1-100": 46.64,
  "Predicted Business Risk Level": "Tier 3: High Risk",
  "Pure Premium Relativity": 16.11
}

